# Mezcla de k listas y Timsort

Mezclar $k$ listas ordenadas con un montículo que guarda la cabeza de cada una cuesta $O(N \log k)$ para $N$ elementos en total, frente a $O(N \log N)$ de concatenar y ordenar. Timsort, el algoritmo de `sorted`, explota la misma idea: busca **rachas** ya ordenadas (invirtiendo las estrictamente descendentes), alarga las cortas hasta un tamaño mínimo *minrun* con inserción binaria y las mezcla de forma equilibrada con una pila. Cuesta $O(n \log n)$ en el peor caso, $O(n)$ con datos ordenados y es estable.

Al final, dos formas de ordenar con otro enfoque: las **redes de ordenación**, una secuencia fija de comparadores que no depende de los datos (se comprueban con el principio 0-1), y la ordenación parcial de los $k$ menores en $O(n \log k)$.

## 1. Mezclar k listas con un montículo

Cada línea de guagua tiene sus horas de salida ordenadas (en minutos desde las 6:00). Escribir un generador `mezclar(listas)` que mantenga en un montículo la cabeza de cada lista `(valor, número de lista, posición)` y saque siempre la menor. Comprobar contra `sorted` de la concatenación, con listas vacías, de un solo elemento y con valores repetidos (los empates salen en el orden de las listas).

In [1]:
import heapq
import random


def mezclar(listas):
    monticulo = [(lista[0], i, 0) for i, lista in enumerate(listas) if lista]
    heapq.heapify(monticulo)
    while monticulo:
        valor, i, j = monticulo[0]
        yield valor
        if j + 1 < len(listas[i]):
            heapq.heapreplace(monticulo, (listas[i][j + 1], i, j + 1))
        else:
            heapq.heappop(monticulo)


lineas = {"L1": [0, 20, 40, 60, 80], "L12": [5, 35, 65], "L91": [], "L25": [20, 21, 95]}
print(list(mezclar(list(lineas.values()))))

rng = random.Random(1)
for _ in range(500):
    listas = [sorted(rng.choices(range(30), k=rng.randint(0, 8))) for _ in range(rng.randint(0, 6))]
    assert list(mezclar(listas)) == sorted(sum(listas, []))

[0, 5, 20, 20, 21, 35, 40, 60, 65, 80, 95]


## 2. `heapq.merge`: perezosa, con `key` y `reverse`

`heapq.merge` hace lo mismo y es perezosa: sirve para iteradores infinitos. Mezclar tres secuencias infinitas de horas de llegada (cada barco llega con su periodo) y tomar las 12 primeras con `islice`. Mezclar también registros ya ordenados por una clave con `key=` y listas descendentes con `reverse=True`, comprobando todo contra `sorted`.

In [2]:
from itertools import count, islice


def llegadas(barco, inicio, periodo):
    for t in count(inicio, periodo):
        yield t, barco


primeras = list(islice(heapq.merge(llegadas("Fred. Olsen", 0, 45), llegadas("Armas", 10, 60), llegadas("Baleària", 30, 90)), 12))
print(", ".join(f"{t // 60:02}:{t % 60:02} {b}" for t, b in primeras))
finitas = [list(islice(llegadas(b, i, p), 12)) for b, i, p in [("Fred. Olsen", 0, 45), ("Armas", 10, 60), ("Baleària", 30, 90)]]
assert primeras == sorted(sum(finitas, []))[:12]

temperaturas = [[("Hierro", 19), ("Gomera", 21), ("Tenerife", 23)], [("Lanzarote", 20), ("Fuerteventura", 22)]]
por_grado = list(heapq.merge(*temperaturas, key=lambda r: r[1]))
print(por_grado)
assert por_grado == sorted(sum(temperaturas, []), key=lambda r: r[1])

descendentes = [[9, 7, 7, 2], [8, 7, 1], [10, 3]]
assert list(heapq.merge(*descendentes, reverse=True)) == sorted(sum(descendentes, []), reverse=True)

00:00 Fred. Olsen, 00:10 Armas, 00:30 Baleària, 00:45 Fred. Olsen, 01:10 Armas, 01:30 Fred. Olsen, 02:00 Baleària, 02:10 Armas, 02:15 Fred. Olsen, 03:00 Fred. Olsen, 03:10 Armas, 03:30 Baleària
[('Hierro', 19), ('Lanzarote', 20), ('Gomera', 21), ('Fuerteventura', 22), ('Tenerife', 23)]


## 3. Cuánto cuesta mezclar

Contar las comparaciones `<` al mezclar $k$ listas ordenadas con $N = 2^{15}$ elementos en total de cuatro formas: el montículo del ejercicio 1, mezclas por parejas en forma de torneo ($\log_2 k$ rondas), mezclar una a una sobre el acumulado ($O(N k)$) y concatenar y llamar a `sorted`. Comparar con $N \log_2 k$ y $N \log_2 N$.

In [3]:
import math


class Contado:
    __slots__ = ("v",)
    comparaciones = 0

    def __init__(self, v):
        self.v = v

    def __lt__(self, otro):
        Contado.comparaciones += 1
        return self.v < otro.v


def mezclar_dos(a, b):
    salida, i, j = [], 0, 0
    while i < len(a) and j < len(b):
        if b[j] < a[i]:
            salida.append(b[j])
            j += 1
        else:
            salida.append(a[i])
            i += 1
    return salida + a[i:] + b[j:]


def torneo(listas):
    while len(listas) > 1:
        listas = [mezclar_dos(*listas[i : i + 2]) if i + 1 < len(listas) else listas[i] for i in range(0, len(listas), 2)]
    return listas[0] if listas else []


def una_a_una(listas):
    acumulado = []
    for lista in listas:
        acumulado = mezclar_dos(acumulado, lista)
    return acumulado


class Cabeza:
    __slots__ = ("valor", "i", "j")

    def __init__(self, valor, i, j):
        self.valor, self.i, self.j = valor, i, j

    def __lt__(self, otra):
        Contado.comparaciones += 1
        return (self.valor.v, self.i) < (otra.valor.v, otra.i)


def mezclar_contando(listas):
    monticulo = [Cabeza(lista[0], i, 0) for i, lista in enumerate(listas) if lista]
    heapq.heapify(monticulo)
    salida = []
    while monticulo:
        c = monticulo[0]
        salida.append(c.valor)
        if c.j + 1 < len(listas[c.i]):
            heapq.heapreplace(monticulo, Cabeza(listas[c.i][c.j + 1], c.i, c.j + 1))
        else:
            heapq.heappop(monticulo)
    return salida


metodos = {
    "montículo": mezclar_contando,
    "torneo": torneo,
    "una a una": una_a_una,
    "sorted": lambda listas: sorted(sum(listas, [])),
}
N = 2**15
rng = random.Random(3)
print(f"{'k':>5}" + "".join(f"{m:>11}" for m in metodos) + f"{'N·log2 k':>10}{'N·log2 N':>10}")
for k in (2, 8, 64, 512):
    valores = [rng.randrange(10**6) for _ in range(N)]
    listas = [sorted(Contado(v) for v in valores[i::k]) for i in range(k)]
    fila = []
    for metodo in metodos.values():
        Contado.comparaciones = 0
        resultado = metodo(listas)
        fila.append(Contado.comparaciones)
        assert [c.v for c in resultado] == sorted(valores)
    print(f"{k:>5}" + "".join(f"{c:>11}" for c in fila) + f"{N * math.log2(k):>10.0f}{N * math.log2(N):>10.0f}")

    k  montículo     torneo  una a una     sorted  N·log2 k  N·log2 N
    2      32766      32766      32766      65537     32768    491520


    8     118975      98287     143321     131063     98304    491520


   64     226550     196474    1062432     229343    196608    491520


  512     325944     293924    8275708     327294    294912    491520


## 4. Rachas naturales

Escribir `racha(a, inicio)`, que devuelve dónde termina la racha que empieza en `inicio`: no descendente ($a_i \le a_{i+1}$) o **estrictamente** descendente ($a_i > a_{i+1}$), en cuyo caso la invierte en el sitio. Partir varias listas en rachas y comprobar que cada trozo queda ordenado. Ver con registros de clave repetida por qué solo se pueden invertir las rachas estrictamente descendentes: invertir una que tenga empates cambia el orden de los iguales.

In [4]:
def racha(a, inicio, menor=lambda x, y: x < y, estricta=True):
    fin = inicio + 1
    if fin == len(a):
        return fin
    if menor(a[fin], a[fin - 1]) or (not estricta and not menor(a[fin - 1], a[fin])):
        fin += 1
        while fin < len(a) and (menor(a[fin], a[fin - 1]) or (not estricta and not menor(a[fin - 1], a[fin]))):
            fin += 1
        a[inicio:fin] = reversed(a[inicio:fin])
    else:
        fin += 1
        while fin < len(a) and not menor(a[fin], a[fin - 1]):
            fin += 1
    return fin


def rachas(a, **opciones):
    a, trozos, i = list(a), [], 0
    while i < len(a):
        fin = racha(a, i, **opciones)
        trozos.append(a[i:fin])
        i = fin
    return trozos


ejemplos = {
    "creciente": [1, 2, 2, 5, 8],
    "decreciente": [9, 7, 4, 1],
    "mezcla": [3, 5, 9, 8, 6, 2, 2, 4, 7, 1],
    "dientes": [1, 4, 7, 2, 5, 8, 3, 6, 9],
}
for nombre, lista in ejemplos.items():
    trozos = rachas(lista)
    print(f"{nombre:<12} {trozos}")
    assert sorted(sum(trozos, [])) == sorted(lista) and all(t == sorted(t) for t in trozos)

registros = [("b", 2), ("a", 2), ("c", 1)]  # dos con clave 2: «b» va antes que «a»
por_clave = lambda x, y: x[1] < y[1]
print("estricta    :", rachas(registros, menor=por_clave))
print("no estricta :", rachas(registros, menor=por_clave, estricta=False))
assert rachas(registros, menor=por_clave, estricta=False)[0][1:] == [("a", 2), ("b", 2)]

creciente    [[1, 2, 2, 5, 8]]
decreciente  [[1, 4, 7, 9]]
mezcla       [[3, 5, 9], [2, 6, 8], [2, 4, 7], [1]]
dientes      [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
estricta    : [[('b', 2), ('a', 2)], [('c', 1)]]
no estricta : [[('c', 1), ('a', 2), ('b', 2)]]


## 5. El tamaño mínimo de racha (*minrun*)

Timsort no deja rachas de menos de *minrun* elementos, un valor entre 32 y 64 elegido para que $n / minrun$ sea una potencia de 2 o un poco menos, y así las mezclas queden equilibradas. Se calcula tomando los 6 bits más altos de $n$ y sumando 1 si alguno de los bits restantes vale 1. Implementarlo, mostrar una tabla para varios $n$ y comprobar esas propiedades para todos los $n$ hasta 100 000.

In [5]:
def calcular_minrun(n):
    resto = 0
    while n >= 64:
        resto |= n & 1
        n >>= 1
    return n + resto


print(f"{'n':>8}{'minrun':>8}{'n / minrun':>12}")
for n in (50, 64, 100, 1000, 2048, 2049, 10_000, 1_000_000):
    print(f"{n:>8}{calcular_minrun(n):>8}{n / calcular_minrun(n):>12.2f}")

for n in range(64, 100_001):
    m = calcular_minrun(n)
    potencia = 1 << (math.ceil(n / m) - 1).bit_length()
    assert 32 <= m <= 64 and potencia // 2 < n / m <= potencia

       n  minrun  n / minrun
      50      50        1.00
      64      32        2.00
     100      50        2.00
    1000      63       15.87
    2048      32       64.00
    2049      33       62.09
   10000      40      250.00
 1000000      62    16129.03


## 6. Alargar las rachas cortas con inserción binaria

Si una racha natural es más corta que *minrun*, Timsort la alarga con los elementos siguientes mediante inserción binaria (busca la posición con `bisect_right`, lo que la mantiene estable, y desplaza). Escribir `insercion_binaria(a, inicio, ordenados, fin)`, que inserta `a[ordenados:fin]` en el tramo ya ordenado `a[inicio:ordenados]`, y partir una lista aleatoria de 1000 elementos en trozos de al menos *minrun*. Comprobar que cada trozo está ordenado.

In [6]:
from bisect import bisect_right


def insercion_binaria(a, inicio, ordenados, fin, clave=lambda x: x):
    for i in range(ordenados, fin):
        x = a[i]
        pos = bisect_right(a, clave(x), inicio, i, key=clave)
        a[pos + 1 : i + 1] = a[pos:i]
        a[pos] = x


rng = random.Random(6)
datos = [rng.randrange(1000) for _ in range(1000)]
a, minrun = list(datos), calcular_minrun(len(datos))
i, longitudes = 0, []
while i < len(a):
    fin = racha(a, i)
    natural = fin - i
    fin_forzado = min(i + minrun, len(a))
    if fin < fin_forzado:
        insercion_binaria(a, i, fin, fin_forzado)
        fin = fin_forzado
    longitudes.append((natural, fin - i))
    assert a[i:fin] == sorted(a[i:fin])
    i = fin
print("minrun:", minrun)
print("(racha natural, tras alargar):", longitudes)
assert sorted(a) == sorted(datos) and all(l == minrun for _, l in longitudes[:-1])

minrun: 63
(racha natural, tras alargar): [(2, 63), (2, 63), (2, 63), (2, 63), (3, 63), (3, 63), (3, 63), (2, 63), (2, 63), (2, 63), (2, 63), (4, 63), (2, 63), (2, 63), (3, 63), (3, 55)]


## 7. Timsort simplificado

Juntar las piezas: recorrer la lista detectando rachas, alargar las cortas hasta *minrun* y apilarlas. Después de cada racha, mezclar las de la cima mientras no se cumplan los invariantes $|Z| > |Y| + |X|$ y $|Y| > |X|$ (siendo $X$ la de la cima), que mantienen la pila con altura logarítmica y las mezclas equilibradas; al final, mezclar todo lo que quede. Sin galope ni otras optimizaciones. Comprobar contra `sorted` con listas de muchas formas.

In [7]:
def timsort(lista, clave=lambda x: x):
    a = list(lista)
    n = len(a)
    estadisticas = {"comparaciones": 0, "rachas": 0, "mezclas": 0, "pila máxima": 0}

    def menor(x, y):
        estadisticas["comparaciones"] += 1
        return clave(x) < clave(y)

    def mezclar_en(i):
        (ini, lx), (_, ly) = pila[i], pila[i + 1]
        izquierda, j, k, destino = a[ini : ini + lx], ini + lx, 0, ini
        while k < lx and j < ini + lx + ly:
            if menor(a[j], izquierda[k]):  # solo pasa la derecha si es estrictamente menor: estable
                a[destino] = a[j]
                j += 1
            else:
                a[destino] = izquierda[k]
                k += 1
            destino += 1
        a[destino : destino + lx - k] = izquierda[k:]
        pila[i : i + 2] = [(ini, lx + ly)]
        estadisticas["mezclas"] += 1

    def colapsar():
        while len(pila) > 1:
            m = len(pila) - 2
            if (m > 0 and pila[m - 1][1] <= pila[m][1] + pila[m + 1][1]) or (
                m > 1 and pila[m - 2][1] <= pila[m - 1][1] + pila[m][1]
            ):
                if pila[m - 1][1] < pila[m + 1][1]:
                    m -= 1
            elif pila[m][1] > pila[m + 1][1]:
                break
            mezclar_en(m)

    minrun = calcular_minrun(n)
    pila, i = [], 0
    while i < n:
        fin = racha(a, i, menor)
        fin_forzado = min(i + minrun, n)
        if fin < fin_forzado:
            for p in range(fin, fin_forzado):  # inserción binaria con la comparación contada
                x, izq, der = a[p], i, p
                while izq < der:
                    medio = (izq + der) // 2
                    if menor(x, a[medio]):
                        der = medio
                    else:
                        izq = medio + 1
                a[izq + 1 : p + 1] = a[izq:p]
                a[izq] = x
            fin = fin_forzado
        pila.append((i, fin - i))
        estadisticas["rachas"] += 1
        estadisticas["pila máxima"] = max(estadisticas["pila máxima"], len(pila))
        colapsar()
        i = fin
    while len(pila) > 1:
        mezclar_en(len(pila) - 2)
    return a, estadisticas


rng = random.Random(7)
formas = {
    "vacía": lambda n: [],
    "aleatoria": lambda n: [rng.randrange(10**6) for _ in range(n)],
    "repetidos": lambda n: [rng.randrange(5) for _ in range(n)],
    "ordenada": lambda n: list(range(n)),
    "inversa": lambda n: list(range(n, 0, -1)),
    "dientes": lambda n: [i % 97 for i in range(n)],
}
for nombre, generar in formas.items():
    for n in (1, 2, 31, 64, 65, 500, 3000):
        datos = generar(n)
        assert timsort(datos)[0] == sorted(datos), (nombre, n)
for _ in range(300):
    datos = [rng.randrange(50) for _ in range(rng.randint(0, 400))]
    assert timsort(datos)[0] == sorted(datos)

ordenada, est = timsort([rng.randrange(10**6) for _ in range(100_000)])
print("100 000 aleatorios:", est)
fi = (1 + 5**0.5) / 2
assert est["pila máxima"] <= math.log(est["rachas"], fi) + 2

100 000 aleatorios: {'comparaciones': 1528609, 'rachas': 2041, 'mezclas': 2040, 'pila máxima': 11}


## 8. Timsort aprovecha el orden

Contar las comparaciones de este Timsort, de un merge sort de arriba abajo y de `sorted` (con la clase `Contado`) en listas de 20 000 elementos aleatorias, ya ordenadas, en orden inverso, casi ordenadas (20 intercambios al azar) y formadas por 10 bloques ordenados. Comprobar que con datos ordenados o inversos Timsort hace exactamente $n - 1$ comparaciones. La ventaja de `sorted` con datos casi ordenados viene del **galope** al mezclar, que esta versión omite.

In [8]:
def merge_sort(a, cuenta):
    if len(a) <= 1:
        return a
    medio = len(a) // 2
    izq, der = merge_sort(a[:medio], cuenta), merge_sort(a[medio:], cuenta)
    salida, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        cuenta[0] += 1
        if der[j] < izq[i]:
            salida.append(der[j])
            j += 1
        else:
            salida.append(izq[i])
            i += 1
    return salida + izq[i:] + der[j:]


def casi_ordenada(n):
    a = list(range(n))
    for _ in range(20):
        i, j = rng.randrange(n), rng.randrange(n)
        a[i], a[j] = a[j], a[i]
    return a


n = 20_000
casos = {
    "aleatoria": [rng.randrange(10**6) for _ in range(n)],
    "ordenada": list(range(n)),
    "inversa": list(range(n, 0, -1)),
    "casi ordenada": casi_ordenada(n),
    "10 bloques": sum((sorted(rng.randrange(10**6) for _ in range(n // 10)) for _ in range(10)), []),
}
print(f"{'datos':<15}{'timsort':>9}{'merge sort':>12}{'sorted':>9}")
for nombre, datos in casos.items():
    resultado, est = timsort(datos)
    cuenta = [0]
    assert resultado == merge_sort(datos, cuenta) == sorted(datos)
    Contado.comparaciones = 0
    sorted(Contado(v) for v in datos)
    print(f"{nombre:<15}{est['comparaciones']:>9}{cuenta[0]:>12}{Contado.comparaciones:>9}")
    if nombre in ("ordenada", "inversa"):
        assert est["comparaciones"] == n - 1 == Contado.comparaciones
print(f"n·log2 n = {n * math.log2(n):.0f}")

datos            timsort  merge sort   sorted
aleatoria         259762      260980   260415
ordenada           19999      139216    19999
inversa            19999      148016    19999


casi ordenada     110054      181811    21956
10 bloques         91972      181176    87990
n·log2 n = 285754


## 9. Estabilidad con claves repetidas

Ordenar 5000 resultados de una carrera (dorsal, categoría, tiempo redondeado al minuto) por tiempo, con muchísimos empates. Comprobar que el Timsort coincide con `sorted(key=...)` y que dentro de cada tiempo se conserva el orden original de los dorsales. Aprovechar la estabilidad para ordenar por categoría y, dentro de ella, por tiempo con dos pasadas (primero la clave secundaria). Ver también que la versión que invierte rachas no estrictas (ejercicio 4) rompe la estabilidad.

In [9]:
categorias = ["sub-23", "sénior", "veterano", "máster"]
corredores = [(dorsal, rng.choice(categorias), rng.randint(180, 240)) for dorsal in range(1, 5001)]
por_tiempo, _ = timsort(corredores, clave=lambda c: c[2])
assert por_tiempo == sorted(corredores, key=lambda c: c[2])
for t in range(180, 241):
    dorsales = [c[0] for c in por_tiempo if c[2] == t]
    assert dorsales == sorted(dorsales)
print("primeros:", por_tiempo[:4])

dos_pasadas, _ = timsort(timsort(corredores, clave=lambda c: c[2])[0], clave=lambda c: c[1])
assert dos_pasadas == sorted(corredores, key=lambda c: (c[1], c[2]))
print("categoría y tiempo:", dos_pasadas[:3])

no_estable = sum(rachas(sorted(corredores, key=lambda c: -c[2]), menor=lambda x, y: x[2] < y[2], estricta=False), [])
iguales = [c[0] for c in no_estable if c[2] == 200]
print("dorsales con 200 min tras invertir una racha no estricta:", iguales[:6], "...")
assert iguales == sorted(iguales, reverse=True)

primeros: [(93, 'sénior', 180), (230, 'máster', 180), (241, 'sénior', 180), (301, 'sénior', 180)]
categoría y tiempo: [(230, 'máster', 180), (392, 'máster', 180), (576, 'máster', 180)]
dorsales con 200 min tras invertir una racha no estricta: [4992, 4969, 4939, 4880, 4856, 4749] ...


## 10. Redes de ordenación: transposición par-impar y bitónica

Una red de ordenación es una lista fija de comparadores $(i, j)$ que dejan el menor en $i$ y el mayor en $j$; no depende de los datos y los comparadores de una misma capa pueden ir en paralelo. Generar la red de **transposición par-impar** ($n$ rondas) y la **bitónica** ($n$ potencia de 2), calcular su tamaño y su profundidad, y verificarlas con el **principio 0-1**: si una red ordena las $2^n$ entradas de ceros y unos, ordena cualquier entrada. Comprobarlo también con todas las permutaciones de 8 elementos.

In [10]:
from itertools import permutations, product


def transposicion(n):
    return [(i, i + 1) for ronda in range(n) for i in range(ronda % 2, n - 1, 2)]


def bitonica(n):
    red, k = [], 2
    while k <= n:
        j = k // 2
        while j >= 1:
            for i in range(n):
                l = i ^ j
                if l > i:
                    red.append((i, l) if i & k == 0 else (l, i))
            j //= 2
        k *= 2
    return red


def aplicar(red, datos):
    a = list(datos)
    for i, j in red:
        if a[j] < a[i]:
            a[i], a[j] = a[j], a[i]
    return a


def profundidad(red, n):
    nivel = [0] * n
    for i, j in red:
        nivel[i] = nivel[j] = max(nivel[i], nivel[j]) + 1
    return max(nivel, default=0)


def ordena_ceros_y_unos(red, n):
    return all(aplicar(red, bits) == sorted(bits) for bits in product((0, 1), repeat=n))


print(f"{'red':<14}{'n':>3}{'comparadores':>14}{'profundidad':>13}{'0-1':>6}")
for n in (4, 8, 16):
    for nombre, generar in [("transposición", transposicion), ("bitónica", bitonica)]:
        red = generar(n)
        ok = ordena_ceros_y_unos(red, n)
        print(f"{nombre:<14}{n:>3}{len(red):>14}{profundidad(red, n):>13}{'sí' if ok else 'no':>6}")
        assert ok
for generar in (transposicion, bitonica):
    red = generar(8)
    assert all(aplicar(red, p) == sorted(p) for p in permutations(range(8)))
print(aplicar(bitonica(8), ["Tejeda", "Agaete", "Mogán", "Gáldar", "Telde", "Artenara", "Arucas", "Firgas"]))

red             n  comparadores  profundidad   0-1
transposición   4             6            4    sí
bitónica        4             6            3    sí
transposición   8            28            8    sí
bitónica        8            24            6    sí


transposición  16           120           16    sí
bitónica       16            80           10    sí


['Agaete', 'Artenara', 'Arucas', 'Firgas', 'Gáldar', 'Mogán', 'Tejeda', 'Telde']


## 11. Mezcla par-impar de Batcher y una red defectuosa

Generar la red de **mezcla par-impar de Batcher** y compararla en tamaño y profundidad con las del ejercicio anterior hasta $n = 32$ (con el principio 0-1 hasta $n = 16$ y entradas aleatorias para 32). Quitar después un comparador a la red de 8 entradas y buscar con el principio 0-1 una entrada de ceros y unos que ya no ordena; comprobar que la misma posición de los ceros y unos da un contraejemplo con números cualesquiera.

In [11]:
def batcher(n):
    red, p = [], 1
    while p < n:
        k = p
        while k >= 1:
            for j in range(k % p, n - k, 2 * k):
                for i in range(min(k, n - j - k)):
                    if (i + j) // (2 * p) == (i + j + k) // (2 * p):
                        red.append((i + j, i + j + k))
            k //= 2
        p *= 2
    return red


print(f"{'n':>3}{'transposición':>15}{'bitónica':>10}{'Batcher':>9}   (comparadores / profundidad)")
for n in (4, 8, 16, 32):
    redes = [transposicion(n), bitonica(n), batcher(n)]
    print(f"{n:>3}" + "".join(f"{f'{len(r)} / {profundidad(r, n)}':>{w}}" for r, w in zip(redes, (15, 10, 9))))
    if n <= 16:
        assert ordena_ceros_y_unos(batcher(n), n)
    else:
        rng_red = random.Random(11)
        for _ in range(2000):
            datos = [rng_red.randrange(100) for _ in range(n)]
            assert aplicar(batcher(n), datos) == sorted(datos)

defectuosa = batcher(8)
quitado = defectuosa.pop(len(defectuosa) // 2)
contraejemplo = next(bits for bits in product((0, 1), repeat=8) if aplicar(defectuosa, bits) != sorted(bits))
numeros = [b * 100 + i for i, b in enumerate(contraejemplo)]
print("comparador quitado:", quitado)
print("0-1:", list(contraejemplo), "->", aplicar(defectuosa, contraejemplo))
print("números:", numeros, "->", aplicar(defectuosa, numeros))
assert aplicar(defectuosa, numeros) != sorted(numeros)

  n  transposición  bitónica  Batcher   (comparadores / profundidad)
  4          6 / 4     6 / 3    5 / 3
  8         28 / 8    24 / 6   19 / 6
 16       120 / 16   80 / 10  63 / 10
 32       496 / 32  240 / 15 191 / 15
comparador quitado: (5, 6)
0-1: [0, 0, 0, 1, 0, 1, 0, 1] -> [0, 0, 0, 0, 1, 0, 1, 1]
números: [0, 1, 2, 103, 4, 105, 6, 107] -> [0, 1, 2, 4, 103, 6, 105, 107]


## 12. Ordenación parcial y `nsmallest`

Para quedarse con los $k$ menores de $n$ no hace falta ordenar todo: basta un montículo **máximo** de tamaño $k$ que se queda con los candidatos ($O(n \log k)$). Implementarlo, compararlo en comparaciones con `heapq.nsmallest` y con `sorted(...)[:k]` para varios $k$, y comprobar que `nsmallest(k, datos, key=...)` equivale a `sorted(datos, key=...)[:k]` también en los empates (es estable). Seleccionar un único elemento sin ordenar nada se ve en *Quick select* (carpeta *Arrays*).

In [12]:
def k_menores(datos, k):
    if k <= 0:
        return []
    monticulo = []  # máximo, guardando -valor (solo para números)
    for x in datos:
        if len(monticulo) < k:
            heapq.heappush(monticulo, -x)
        elif x < -monticulo[0]:
            heapq.heapreplace(monticulo, -x)
    return sorted(-v for v in monticulo)


class Negable(Contado):
    __slots__ = ()

    def __neg__(self):
        return Negable(-self.v)


n = 100_000
valores = [rng.randrange(10**7) for _ in range(n)]
contados = [Negable(v) for v in valores]
print(f"{'k':>6}{'k_menores':>11}{'nsmallest':>11}{'sorted[:k]':>12}{'n·log2 k':>10}")
for k in (1, 10, 100, 1000, 10_000):
    fila = []
    for metodo in (k_menores, lambda d, k: heapq.nsmallest(k, d), lambda d, k: sorted(d)[:k]):
        Contado.comparaciones = 0
        resultado = metodo(contados, k)
        fila.append(Contado.comparaciones)
        assert [c.v for c in resultado] == sorted(valores)[:k]
    print(f"{k:>6}" + "".join(f"{c:>{w}}" for c, w in zip(fila, (11, 11, 12))) + f"{n * math.log2(max(k, 2)):>10.0f}")

jugadores = [(f"j{i:02}", rng.randint(1, 6)) for i in range(40)]
for k in (1, 5, 12):
    assert heapq.nsmallest(k, jugadores, key=lambda j: j[1]) == sorted(jugadores, key=lambda j: j[1])[:k]
    assert heapq.nlargest(k, jugadores, key=lambda j: j[1]) == sorted(jugadores, key=lambda j: j[1], reverse=True)[:k]
print("los 5 con menos puntos:", heapq.nsmallest(5, jugadores, key=lambda j: j[1]))

     k  k_menores  nsmallest  sorted[:k]  n·log2 k


     1      99999      99999     1531992    100000


    10     100400     100397     1531992    332193


   100     106043     105944     1531992    664386


  1000     159767     159115     1531992    996578


 10000     558657     552504     1531992   1328771
los 5 con menos puntos: [('j03', 1), ('j16', 1), ('j19', 1), ('j25', 1), ('j26', 1)]
